In [1]:
# ============================================================
# Cell 1 — Thunder setup + Kaggle downloads
# Non-root safe version
# ============================================================

!pip install -q kaggle timm albumentations tqdm opencv-python-headless seaborn scikit-learn matplotlib

import os, zipfile, shutil
from pathlib import Path

# ----------------------------
# Kaggle API credentials — non-root safe
# ----------------------------
HOME = Path.home()
KAGGLE_DIR = HOME / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

with open(KAGGLE_DIR / "kaggle.json", "w") as f:
    f.write('{"username":"ahnafnafim","key":"YOUR_KEY"}')  # replace YOUR_KEY

os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)

# Make sure Kaggle CLI reads from this location
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)

print("Kaggle config saved to:", KAGGLE_DIR / "kaggle.json")
print("Running as home:", HOME)

# Test authentication
!kaggle datasets list -m

# ----------------------------
# Local Thunder folders
# ----------------------------
ASSET_ROOT = HOME / "deepfake_assets"
ASSET_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = ASSET_ROOT / "convnext-checkpoint"
CKPT_ROOT    = ASSET_ROOT / "convnext-checkpoints"
TGIF_PARENT  = ASSET_ROOT / "tgif"

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
TGIF_PARENT.mkdir(parents=True, exist_ok=True)

print("\nASSET_ROOT:", ASSET_ROOT)

# ----------------------------
# Download your uploaded cross-domain dataset
# ----------------------------
if not any(DATASET_ROOT.iterdir()):
    print("\nDownloading cross-domain datasets...")
    !kaggle datasets download -d ahnafnafim/convnext-checkpoint -p "{ASSET_ROOT}" -q

    zip_path = ASSET_ROOT / "convnext-checkpoint.zip"

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATASET_ROOT)
        zip_path.unlink()
        print("✓ Extracted cross-domain datasets to:", DATASET_ROOT)
    else:
        print("⚠ convnext-checkpoint.zip not found. Check dataset slug.")
else:
    print("✓ Cross-domain datasets already exist:", DATASET_ROOT)

# ----------------------------
# Download your uploaded checkpoints
# ----------------------------
if not any(CKPT_ROOT.glob("*.pth")):
    print("\nDownloading checkpoints...")
    !kaggle datasets download -d ahnafnafim/convnext-checkpoints -p "{ASSET_ROOT}" -q

    zip_path = ASSET_ROOT / "convnext-checkpoints.zip"

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(CKPT_ROOT)
        zip_path.unlink()
        print("✓ Extracted checkpoints to:", CKPT_ROOT)
    else:
        print("⚠ convnext-checkpoints.zip not found. Check dataset slug.")
else:
    print("✓ Checkpoints already exist:", CKPT_ROOT)

# ----------------------------
# Download TGIF subset
# ----------------------------
TGIF_ROOT = TGIF_PARENT / "subset"

if not TGIF_ROOT.exists():
    print("\nDownloading TGIF subset...")
    !kaggle datasets download -d araftahsanpavel/tgif-subset -p "{TGIF_PARENT}" -q

    zip_path = TGIF_PARENT / "tgif-subset.zip"

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(TGIF_PARENT)
        zip_path.unlink()
        print("✓ TGIF extracted to:", TGIF_ROOT)
    else:
        print("⚠ TGIF zip not found.")
else:
    print("✓ TGIF already exists:", TGIF_ROOT)

# ----------------------------
# Show final structure
# ----------------------------
print("\n=== Final paths ===")
print("TGIF_ROOT       :", TGIF_ROOT, TGIF_ROOT.exists())
print("DATASET_ROOT    :", DATASET_ROOT, DATASET_ROOT.exists())
print("CKPT_ROOT       :", CKPT_ROOT, CKPT_ROOT.exists())

print("\nCheckpoints:")
for p in CKPT_ROOT.rglob("*.pth"):
    print(" ", p, f"{p.stat().st_size / 1024**2:.1f} MB")

print("\nCross-domain top-level folders:")
for p in DATASET_ROOT.iterdir():
    if p.is_dir():
        print(" ", p)

Kaggle config saved to: /home/ubuntu/.kaggle/kaggle.json
Running as home: /home/ubuntu
No datasets found

ASSET_ROOT: /home/ubuntu/deepfake_assets

Dataset URL: https://www.kaggle.com/datasets/ahnafnafim/convnext-checkpoint
License(s): unknown
✓ Extracted cross-domain datasets to: /home/ubuntu/deepfake_assets/convnext-checkpoint

Dataset URL: https://www.kaggle.com/datasets/ahnafnafim/convnext-checkpoints
License(s): unknown
✓ Extracted checkpoints to: /home/ubuntu/deepfake_assets/convnext-checkpoints

Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
✓ TGIF extracted to: /home/ubuntu/deepfake_assets/tgif/subset

=== Final paths ===
TGIF_ROOT       : /home/ubuntu/deepfake_assets/tgif/subset True
DATASET_ROOT    : /home/ubuntu/deepfake_assets/convnext-checkpoint True
CKPT_ROOT       : /home/ubuntu/deepfake_assets/convnext-checkpoints True

Checkpoints:
  /home/ubuntu/deepfake_assets/convnext-checkpoints/best_full_novelty_aug.pth 762.1 MB
  /hom

In [1]:
# ============================================================
# Cell 2 — Imports + paths + config
# ============================================================

from __future__ import annotations

import os, time, random, json, glob, shutil, gc
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ----------------------------
# Runtime settings
# ----------------------------
IMAGE_SIZE = 384
BATCH_SIZE = 8
BINARY_THRESHOLD = 0.5
SEED = 42
NUM_WORKERS = 4
PIN_MEMORY = False

# Small defaults for stability
GRADCAM_SAMPLES = 4
FAITHFULNESS_SAMPLES = 10

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ----------------------------
# Thunder paths
# ----------------------------
HOME = Path.home()
ASSET_ROOT = HOME / "deepfake_assets"

DATA_ROOT = ASSET_ROOT / "tgif" / "subset"

CROSS_DATA_ROOT = ASSET_ROOT / "convnext-checkpoint"
CHECKPOINT_ROOT = ASSET_ROOT / "convnext-checkpoints"

AUTOSPLICE_ROOT = CROSS_DATA_ROOT / "AutoSplice" / "AutoSplice"
CASIA_ROOT      = CROSS_DATA_ROOT / "CASIA_v2" / "casia"
IMD2020_ROOT    = CROSS_DATA_ROOT / "IMD2020"

SAVE_DIR = HOME / "xai_results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nPaths:")
print("DATA_ROOT       :", DATA_ROOT, DATA_ROOT.exists())
print("AUTOSPLICE_ROOT :", AUTOSPLICE_ROOT, AUTOSPLICE_ROOT.exists())
print("CASIA_ROOT      :", CASIA_ROOT, CASIA_ROOT.exists())
print("IMD2020_ROOT    :", IMD2020_ROOT, IMD2020_ROOT.exists())
print("CHECKPOINT_ROOT :", CHECKPOINT_ROOT, CHECKPOINT_ROOT.exists())
print("SAVE_DIR        :", SAVE_DIR)

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA RTX A6000

Paths:
DATA_ROOT       : /home/ubuntu/deepfake_assets/tgif/subset True
AUTOSPLICE_ROOT : /home/ubuntu/deepfake_assets/convnext-checkpoint/AutoSplice/AutoSplice True
CASIA_ROOT      : /home/ubuntu/deepfake_assets/convnext-checkpoint/CASIA_v2/casia True
IMD2020_ROOT    : /home/ubuntu/deepfake_assets/convnext-checkpoint/IMD2020 True
CHECKPOINT_ROOT : /home/ubuntu/deepfake_assets/convnext-checkpoints True
SAVE_DIR        : /home/ubuntu/xai_results


In [2]:
# ============================================================
# Cell 3 — Model definitions
# ============================================================

class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()

        f1 = np.array([
            [0,0,0,0,0],
            [0,-1,2,-1,0],
            [0,2,-4,2,0],
            [0,-1,2,-1,0],
            [0,0,0,0,0]
        ], dtype=np.float32) / 4.0

        f2 = np.array([
            [-1,2,-2,2,-1],
            [2,-6,8,-6,2],
            [-2,8,-12,8,-2],
            [2,-6,8,-6,2],
            [-1,2,-2,2,-1]
        ], dtype=np.float32) / 12.0

        f3 = np.array([
            [0,0,0,0,0],
            [0,0,0,0,0],
            [0,1,-2,1,0],
            [0,0,0,0,0],
            [0,0,0,0,0]
        ], dtype=np.float32) / 2.0

        self.register_buffer("weight", torch.from_numpy(np.stack([f1, f2, f3])[:, None]))

    def forward(self, x):
        return torch.cat(
            [F.conv2d(x[:, c:c+1], self.weight, padding=2) for c in range(3)],
            dim=1
        )


class BayarConv2d(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, k=5):
        super().__init__()
        self.k = k
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.01)

    def forward(self, x):
        w = self.weight.clone()
        c = self.k // 2
        wn = w.clone()
        wn[:, :, c, c] = 0.0
        w[:, :, c, c] = -wn.sum(dim=[2, 3])
        return F.conv2d(x, w, padding=c)


class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)

        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, mid),
            nn.ReLU(),
            nn.Linear(mid, ch),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x * self.ca(x).view(x.size(0), -1, 1, 1)
        return x * self.sa(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1))


class FPNBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )

        self.cbam = CBAM(out_ch)

    def forward(self, x, skip=None):
        x = self.up(x)

        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
            x = torch.cat([x, skip], dim=1)

        return self.cbam(self.conv(x))


class NoiseBranch(nn.Module):
    def __init__(self, out_ch=128):
        super().__init__()

        self.srm = SRMConv()
        self.bayar = BayarConv2d(3, 3)

        self.enc = nn.Sequential(
            nn.Conv2d(12, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.enc(torch.cat([self.srm(x), self.bayar(x)], dim=1))


class DCTStream(nn.Module):
    def __init__(self, out_channels=128):
        super().__init__()

        self.freq_conv = nn.Sequential(
            nn.Conv2d(3, 32, 8, stride=8, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.Conv2d(64, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.freq_conv(x)


class SEFusionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, reduction=16):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True)
        )

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_ch, max(out_ch // reduction, 4), bias=False),
            nn.ReLU(True),
            nn.Linear(max(out_ch // reduction, 4), out_ch, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.proj(x)
        return x * self.se(x).view(x.shape[0], x.shape[1], 1, 1)


class ContrastiveHead(nn.Module):
    def __init__(self, in_ch, proj_dim=128):
        super().__init__()

        self.proj = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_ch, in_ch, bias=False),
            nn.ReLU(True),
            nn.Linear(in_ch, proj_dim, bias=False)
        )

    def forward(self, x):
        return F.normalize(self.proj(x), dim=1)


class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True)
        )

    def forward(self, x, skip=None):
        x = self.up(x)

        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
            x = torch.cat([x, skip], dim=1)

        return self.conv(x)


class ConvNeXtLUNet(nn.Module):
    def __init__(self, image_size=IMAGE_SIZE, pretrained=False):
        super().__init__()

        self.image_size = image_size

        self.enc = timm.create_model(
            "convnext_large",
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )

        self.lat4 = nn.Conv2d(1536, 256, 1)

        self.dec3 = UNetDecoderBlock(256, 768, 256)
        self.dec2 = UNetDecoderBlock(256, 384, 128)
        self.dec1 = UNetDecoderBlock(128, 192, 64)
        self.dec0 = UNetDecoderBlock(64, 0, 32)

        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        s1, s2, s3, s4 = self.enc(x)

        p4 = self.lat4(s4)
        d3 = self.dec3(p4, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        d0 = self.dec0(d1)

        seg = self.head(d0)

        if seg.shape[2:] != (self.image_size, self.image_size):
            seg = F.interpolate(
                seg,
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False
            )

        return seg


class EnhancedForensicNet(nn.Module):
    def __init__(self, image_size=IMAGE_SIZE, pretrained=False):
        super().__init__()

        self.image_size = image_size

        self.enc = timm.create_model(
            "convnext_large",
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )

        self.noise = NoiseBranch(128)
        self.dct = DCTStream(128)

        self.lat4 = SEFusionBlock(1536 + 128 + 128, 256)

        self.lat3 = nn.Conv2d(768, 256, 1)
        self.lat2 = nn.Conv2d(384, 128, 1)
        self.lat1 = nn.Conv2d(192, 64, 1)

        self.dec3 = FPNBlock(256, 256, 256)
        self.dec2 = FPNBlock(256, 128, 128)
        self.dec1 = FPNBlock(128, 64, 64)
        self.dec0 = FPNBlock(64, 0, 32)

        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )

        self.edge_head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )

        self.deep_head3 = nn.Conv2d(256, 1, 1)
        self.deep_head2 = nn.Conv2d(128, 1, 1)
        self.deep_head1 = nn.Conv2d(64, 1, 1)

        self.contrast_head = ContrastiveHead(256, 128)

    def forward(self, x):
        s1, s2, s3, s4 = self.enc(x)

        n_feat = F.adaptive_avg_pool2d(self.noise(x), s4.shape[2:])
        dct_feat = F.adaptive_avg_pool2d(self.dct(x), s4.shape[2:])

        p4 = self.lat4(torch.cat([s4, n_feat, dct_feat], dim=1))

        d3 = self.dec3(p4, self.lat3(s3))
        d2 = self.dec2(d3, self.lat2(s2))
        d1 = self.dec1(d2, self.lat1(s1))
        d0 = self.dec0(d1)

        seg = self.head(d0)

        if seg.shape[2:] != (self.image_size, self.image_size):
            seg = F.interpolate(
                seg,
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False
            )

        edge = self.edge_head(d0)

        if edge.shape[2:] != (self.image_size, self.image_size):
            edge = F.interpolate(
                edge,
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False
            )

        return {
            "mask": seg,
            "edge": edge,
            "deep": [self.deep_head3(d3), self.deep_head2(d2), self.deep_head1(d1)],
            "embed": self.contrast_head(p4),
        }


print("✓ Models defined")

✓ Models defined


In [3]:
# ============================================================
# Cell 4 — Checkpoint loading utilities
# ============================================================

def clean_state_dict_keys(state):
    """
    Handles checkpoints saved with DataParallel/module prefix.
    """
    if not isinstance(state, dict):
        return state

    new_state = {}
    for k, v in state.items():
        if k.startswith("module."):
            new_state[k.replace("module.", "", 1)] = v
        else:
            new_state[k] = v
    return new_state


def load_state_flexible(model, ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    if isinstance(ckpt, dict):
        if "model" in ckpt:
            state = ckpt["model"]
        elif "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
        else:
            state = ckpt
    else:
        state = ckpt

    state = clean_state_dict_keys(state)
    model.load_state_dict(state)
    model.eval()
    return model


novelty_ckpt_path = CHECKPOINT_ROOT / "best_full_novelty_aug.pth"
baseline_ckpt_path = CHECKPOINT_ROOT / "best_base_1_convnextl_unet.pth"

print("Novelty checkpoint:", novelty_ckpt_path, novelty_ckpt_path.exists())
print("Baseline checkpoint:", baseline_ckpt_path, baseline_ckpt_path.exists())

if not novelty_ckpt_path.exists():
    raise FileNotFoundError(f"Missing novelty checkpoint: {novelty_ckpt_path}")

if not baseline_ckpt_path.exists():
    raise FileNotFoundError(f"Missing baseline checkpoint: {baseline_ckpt_path}")


def load_model_by_name(model_key):
    if model_key == "baseline":
        model = ConvNeXtLUNet(pretrained=False).to(device)
        model = load_state_flexible(model, baseline_ckpt_path, device)
        model_name = "ConvNeXt-L U-Net baseline"

    elif model_key == "novelty":
        model = EnhancedForensicNet(pretrained=False).to(device)
        model = load_state_flexible(model, novelty_ckpt_path, device)
        model_name = "Full novelty augmented"

    else:
        raise ValueError("model_key must be 'baseline' or 'novelty'")

    return model, model_name


def clear_model(model):
    del model
    torch.cuda.empty_cache()
    gc.collect()


print("✓ Checkpoint utilities ready")

Novelty checkpoint: /home/ubuntu/deepfake_assets/convnext-checkpoints/best_full_novelty_aug.pth True
Baseline checkpoint: /home/ubuntu/deepfake_assets/convnext-checkpoints/best_base_1_convnextl_unet.pth True
✓ Checkpoint utilities ready


In [4]:
# ============================================================
# Cell 5 — Dataset indexing
# ============================================================

val_tf = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def is_image_file(p: Path):
    return p.suffix.lower() in IMG_EXTS

def unnormalize(tensor):
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

def normalize_stem(stem: str):
    s = stem.lower()
    remove_tokens = [
        "_mask", "-mask", "_gt", "-gt",
        "_groundtruth", "_ground_truth",
        "_ps_mask", "_copy", "_fake",
        "_tampered", "_forged", "_label"
    ]
    for t in remove_tokens:
        s = s.replace(t, "")
    return s

def is_mask_like_path(p: Path):
    s = str(p).lower()
    return any(k in s for k in ["mask", "gt", "groundtruth", "ground_truth", "label"])

def find_matching_mask(img_path: Path, mask_files):
    img_stem = normalize_stem(img_path.stem)

    for m in mask_files:
        if normalize_stem(m.stem) == img_stem:
            return m

    for m in mask_files:
        ms = normalize_stem(m.stem)
        if img_stem in ms or ms in img_stem:
            return m

    return None

def is_probably_binary_mask(p: Path):
    try:
        arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if arr is None:
            return False
        small = cv2.resize(arr, (64, 64), interpolation=cv2.INTER_NEAREST)
        uniq = np.unique(small)
        return len(uniq) <= 10
    except Exception:
        return False


# ----------------------------
# TGIF
# ----------------------------
def build_tgif_index(split_dir: Path):
    samples = []
    images_root = split_dir / "images"
    masks_root = split_dir / "masks"
    fakes_root = split_dir / "fakes"

    if fakes_root.exists():
        for p in fakes_root.rglob("*.png"):
            if "_mask_" not in p.name:
                continue

            name = p.name
            category = p.parent.name
            stem = name

            for tag in ["_sd2_", "_sdxl_", "_firefly_", "_dalle_", "_dalle2_"]:
                if tag in stem:
                    stem = stem.split(tag)[0] + ".png"
                    break

            mask_path = masks_root / category / stem

            if not mask_path.exists() and (masks_root / category).exists():
                for m in (masks_root / category).glob("*.png"):
                    if name.startswith(m.name):
                        mask_path = m
                        break

            if mask_path.exists():
                samples.append({
                    "image_path": str(p),
                    "mask_path": str(mask_path),
                    "is_fake": True,
                    "dataset": "TGIF"
                })

    if images_root.exists():
        for p in images_root.rglob("*_orig.png"):
            samples.append({
                "image_path": str(p),
                "mask_path": None,
                "is_fake": False,
                "dataset": "TGIF"
            })

    return samples


# ----------------------------
# AutoSplice
# ----------------------------
def build_autosplice_index(root: Path, jpeg_variant="Forged_JPEG100"):
    samples = []

    mask_dir = root / "Mask"
    forged_dir = root / jpeg_variant

    if not forged_dir.exists():
        print(f"⚠ Missing AutoSplice forged dir: {forged_dir}")
        return samples

    if not mask_dir.exists():
        print(f"⚠ Missing AutoSplice mask dir: {mask_dir}")
        return samples

    mask_files = [p for p in mask_dir.rglob("*") if p.is_file() and is_image_file(p)]

    for img_path in sorted(forged_dir.rglob("*")):
        if not img_path.is_file() or not is_image_file(img_path):
            continue

        mask_path = find_matching_mask(img_path, mask_files)

        if mask_path is not None:
            samples.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path),
                "is_fake": True,
                "dataset": f"AutoSplice_{jpeg_variant}"
            })

    return samples


# ----------------------------
# CASIA v2
# ----------------------------
def build_casia_index(root: Path):
    samples = []

    tp_dir = root / "Tp"
    gt_dir = root / "Gt"

    if not tp_dir.exists():
        print(f"⚠ Missing CASIA Tp dir: {tp_dir}")
        return samples

    if not gt_dir.exists():
        print(f"⚠ Missing CASIA Gt dir: {gt_dir}")
        return samples

    mask_files = [p for p in gt_dir.rglob("*") if p.is_file() and is_image_file(p)]

    for img_path in sorted(tp_dir.rglob("*")):
        if not img_path.is_file() or not is_image_file(img_path):
            continue

        mask_path = find_matching_mask(img_path, mask_files)

        if mask_path is not None:
            samples.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path),
                "is_fake": True,
                "dataset": "CASIA v2"
            })

    return samples


# ----------------------------
# IMD2020
# ----------------------------
def build_imd2020_index(root: Path, max_folders=300):
    samples = []

    if not root.exists():
        print(f"⚠ Missing IMD2020 root: {root}")
        return samples

    leaf_dirs = []

    for d in root.rglob("*"):
        if d.is_dir():
            try:
                image_files = [p for p in d.iterdir() if p.is_file() and is_image_file(p)]
                if len(image_files) >= 2:
                    leaf_dirs.append(d)
            except Exception:
                continue

    if max_folders is not None:
        leaf_dirs = leaf_dirs[:max_folders]

    for d in tqdm(leaf_dirs, desc="Indexing IMD2020 folders", leave=False):
        files = [p for p in d.iterdir() if p.is_file() and is_image_file(p)]

        mask_candidates = [p for p in files if is_mask_like_path(p)]

        if not mask_candidates:
            mask_candidates = [p for p in files if is_probably_binary_mask(p)]

        if not mask_candidates:
            continue

        mask_path = mask_candidates[0]

        image_candidates = [p for p in files if p != mask_path and not is_probably_binary_mask(p)]

        if not image_candidates:
            image_candidates = [p for p in files if p != mask_path]

        if not image_candidates:
            continue

        img_path = image_candidates[0]

        samples.append({
            "image_path": str(img_path),
            "mask_path": str(mask_path),
            "is_fake": True,
            "dataset": "IMD2020"
        })

    return samples


# ----------------------------
# Dataset class
# ----------------------------
class GenericDataset(Dataset):
    def __init__(self, samples, image_size=IMAGE_SIZE, transform=None, forged_only=True):
        if forged_only:
            self.samples = [s for s in samples if s.get("is_fake", True)]
        else:
            self.samples = samples

        self.transform = transform or val_tf
        self.image_size = image_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        img = cv2.imread(s["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(s["image_path"])

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if s.get("mask_path") is not None:
            mask = cv2.imread(s["mask_path"], cv2.IMREAD_GRAYSCALE)

            if mask is None:
                mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
            else:
                # Convert to binary first
                mask = (mask >= 127).astype(np.uint8)

                # IMPORTANT FIX:
                # Make mask H,W match image H,W before Albumentations.
                if mask.shape[:2] != img.shape[:2]:
                    mask = cv2.resize(
                        mask,
                        (img.shape[1], img.shape[0]),  # width, height
                        interpolation=cv2.INTER_NEAREST
                    )
        else:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        out = self.transform(image=img, mask=mask)

        return {
            "image": out["image"].float(),
            "mask": out["mask"].float().unsqueeze(0),
            "path": s["image_path"],
            "mask_path": s.get("mask_path", None),
        }

# ----------------------------
# Build datasets
# ----------------------------
print("Building dataset indices...")

tgif_test_samples = build_tgif_index(DATA_ROOT / "testing") if DATA_ROOT.exists() else []
autosplice_samples = build_autosplice_index(AUTOSPLICE_ROOT, jpeg_variant="Forged_JPEG100")
casia_samples = build_casia_index(CASIA_ROOT)

# Keep capped first for stability. Later, set max_folders=None for full IMD2020.
imd2020_samples = build_imd2020_index(IMD2020_ROOT, max_folders=300)

print(f"TGIF test     : {len(tgif_test_samples)}")
print(f"AutoSplice    : {len(autosplice_samples)}")
print(f"CASIA v2      : {len(casia_samples)}")
print(f"IMD2020       : {len(imd2020_samples)}")

datasets = {
    "TGIF": GenericDataset(tgif_test_samples, forged_only=True) if tgif_test_samples else None,
    "AutoSplice": GenericDataset(autosplice_samples, forged_only=True) if autosplice_samples else None,
    "CASIA v2": GenericDataset(casia_samples, forged_only=True) if casia_samples else None,
    "IMD2020": GenericDataset(imd2020_samples, forged_only=True) if imd2020_samples else None,
}

for name, ds in datasets.items():
    print(f"{name:<12}: {len(ds) if ds is not None else 0}")

Building dataset indices...


Indexing IMD2020 folders:  43%|████▎     | 128/300 [00:11<00:14, 11.60it/s]libpng warning: iCCP: duplicate
libpng warning: iCCP: duplicate
libpng warning: iCCP: duplicate
libpng warning: iCCP: duplicate
Indexing IMD2020 folders:  67%|██████▋   | 200/300 [00:18<00:08, 11.65it/s]libpng warning: iCCP: profile 'ICC profile': 'desc': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'wtpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bkpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'rXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'gXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmnd': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmdd': ICC 

TGIF test     : 1029
AutoSplice    : 3621
CASIA v2      : 5123
IMD2020       : 300
TGIF        : 686
AutoSplice  : 3621
CASIA v2    : 5123
IMD2020     : 300


In [5]:
# ============================================================
# Cell 6 — Streaming metric evaluation
# ============================================================

class StreamingSegMetrics:
    def __init__(self):
        self.tp = 0
        self.fp = 0
        self.tn = 0
        self.fn = 0

    def update(self, pred, gt):
        pred = pred.astype(bool)
        gt = gt.astype(bool)

        self.tp += int(np.logical_and(pred, gt).sum())
        self.fp += int(np.logical_and(pred, np.logical_not(gt)).sum())
        self.tn += int(np.logical_and(np.logical_not(pred), np.logical_not(gt)).sum())
        self.fn += int(np.logical_and(np.logical_not(pred), gt).sum())

    def compute(self):
        eps = 1e-8

        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        iou = self.tp / (self.tp + self.fp + self.fn + eps)
        accuracy = (self.tp + self.tn) / (self.tp + self.fp + self.tn + self.fn + eps)

        return {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "iou": float(iou),
            "tp": self.tp,
            "fp": self.fp,
            "tn": self.tn,
            "fn": self.fn,
        }


@torch.no_grad()
def evaluate_model_streaming(model, dataset, dataset_name, model_name, max_samples=None):
    if dataset is None or len(dataset) == 0:
        print(f"⚠ Skipping {dataset_name}: no samples")
        return None

    if max_samples is not None:
        indices = list(range(min(max_samples, len(dataset))))
        subset = torch.utils.data.Subset(dataset, indices)
    else:
        subset = dataset

    loader = DataLoader(
        subset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    model.eval()
    metric = StreamingSegMetrics()

    for batch in tqdm(loader, desc=f"Eval {model_name} on {dataset_name}", leave=False):
        images = batch["image"].to(device)
        masks = batch["mask"].numpy()

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            output = model(images)

        logits = output["mask"] if isinstance(output, dict) else output
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        pred = (probs[0, 0] > BINARY_THRESHOLD).astype(np.uint8)
        gt = masks[0, 0].astype(np.uint8)

        metric.update(pred, gt)

        del images, output, logits, probs

    out = metric.compute()
    out["dataset"] = dataset_name
    out["model"] = model_name
    out["n_samples"] = len(subset)

    return out

print("✓ Streaming evaluator ready")

✓ Streaming evaluator ready


In [10]:
# ============================================================
# Cell 7 — Baseline cross-domain evaluation
# ============================================================

model, model_name = load_model_by_name("baseline")

baseline_results = {}

for dataset_name, dataset in datasets.items():
    if dataset is None or len(dataset) == 0:
        continue

    max_samples = 300 if dataset_name == "IMD2020" else None

    metrics = evaluate_model_streaming(
        model=model,
        dataset=dataset,
        dataset_name=dataset_name,
        model_name=model_name,
        max_samples=max_samples
    )

    baseline_results[dataset_name] = metrics

    print(
        f"{dataset_name:<12} | {model_name:<28} | "
        f"IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | "
        f"Prec: {metrics['precision']:.4f} | Rec: {metrics['recall']:.4f} | "
        f"N: {metrics['n_samples']}"
    )

with open(SAVE_DIR / "baseline_cross_domain_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2, default=str)

print("✓ Saved baseline results")

clear_model(model)

TGIF         | ConvNeXt-L U-Net baseline    | IoU: 0.8306 | F1: 0.9075 | Prec: 0.9177 | Rec: 0.8975 | N: 686


AutoSplice   | ConvNeXt-L U-Net baseline    | IoU: 0.0444 | F1: 0.0850 | Prec: 0.9150 | Rec: 0.0446 | N: 3621


Eval ConvNeXt-L U-Net baseline on CASIA v2:  50%|█████     | 2566/5123 [09:18<10:23,  4.10it/s][ WARN:0@3967.031] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered


CASIA v2     | ConvNeXt-L U-Net baseline    | IoU: 0.0101 | F1: 0.0200 | Prec: 0.7023 | Rec: 0.0102 | N: 5123


Eval ConvNeXt-L U-Net baseline on IMD2020:  42%|████▏     | 125/300 [00:32<00:49,  3.52it/s]libpng warning: iCCP: profile 'ICC profile': 'desc': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'wtpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bkpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'rXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'gXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmnd': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmdd': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'vued': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'view': 

IMD2020      | ConvNeXt-L U-Net baseline    | IoU: 0.0154 | F1: 0.0303 | Prec: 0.2208 | Rec: 0.0162 | N: 300
✓ Saved baseline results


In [11]:
# ============================================================
# Cell 8 — Novelty cross-domain evaluation
# ============================================================

model, model_name = load_model_by_name("novelty")

novelty_results = {}

for dataset_name, dataset in datasets.items():
    if dataset is None or len(dataset) == 0:
        continue

    max_samples = 300 if dataset_name == "IMD2020" else None

    metrics = evaluate_model_streaming(
        model=model,
        dataset=dataset,
        dataset_name=dataset_name,
        model_name=model_name,
        max_samples=max_samples
    )

    novelty_results[dataset_name] = metrics

    print(
        f"{dataset_name:<12} | {model_name:<28} | "
        f"IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | "
        f"Prec: {metrics['precision']:.4f} | Rec: {metrics['recall']:.4f} | "
        f"N: {metrics['n_samples']}"
    )

with open(SAVE_DIR / "novelty_cross_domain_results.json", "w") as f:
    json.dump(novelty_results, f, indent=2, default=str)

print("✓ Saved novelty results")

clear_model(model)

TGIF         | Full novelty augmented       | IoU: 0.8790 | F1: 0.9356 | Prec: 0.9471 | Rec: 0.9244 | N: 686


AutoSplice   | Full novelty augmented       | IoU: 0.0622 | F1: 0.1171 | Prec: 0.9400 | Rec: 0.0624 | N: 3621


Eval Full novelty augmented on CASIA v2:  50%|█████     | 2566/5123 [10:25<08:33,  4.98it/s][ WARN:0@6339.800] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered


CASIA v2     | Full novelty augmented       | IoU: 0.0102 | F1: 0.0203 | Prec: 0.7454 | Rec: 0.0103 | N: 5123


Eval Full novelty augmented on IMD2020:  42%|████▏     | 125/300 [00:36<00:42,  4.07it/s]libpng warning: iCCP: profile 'ICC profile': 'desc': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'wtpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bkpt': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'rXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'gXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'bXYZ': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmnd': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'dmdd': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'vued': ICC profile tag start not a multiple of 4
libpng warning: iCCP: profile 'ICC profile': 'view': ICC

IMD2020      | Full novelty augmented       | IoU: 0.0159 | F1: 0.0313 | Prec: 0.2586 | Rec: 0.0167 | N: 300
✓ Saved novelty results


In [12]:
# ============================================================
# Cell 9 — Combine cross-domain results
# ============================================================

combined_results = {
    "baseline": baseline_results if "baseline_results" in globals() else None,
    "novelty": novelty_results if "novelty_results" in globals() else None,
    "notes": {
        "best_model": "best_full_novelty_aug.pth",
        "best_model_known_test_iou": 0.8742,
        "baseline": "best_base_1_convnextl_unet.pth",
        "imd2020_note": "IMD2020 capped to 300 indexed/evaluated samples initially for stability."
    },
    "paths": {
        "tgif": str(DATA_ROOT),
        "autosplice": str(AUTOSPLICE_ROOT),
        "casia": str(CASIA_ROOT),
        "imd2020": str(IMD2020_ROOT),
        "checkpoints": str(CHECKPOINT_ROOT),
    }
}

with open(SAVE_DIR / "cross_domain_combined_results.json", "w") as f:
    json.dump(combined_results, f, indent=2, default=str)

print("✓ Saved combined cross-domain results:")
print(SAVE_DIR / "cross_domain_combined_results.json")

✓ Saved combined cross-domain results:
/home/ubuntu/xai_results/cross_domain_combined_results.json


In [13]:
# ============================================================
# Cell 10 — GradCAM utilities
# ============================================================

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    @torch.enable_grad()
    def generate(self, input_tensor, target_mask=None):
        self.model.eval()
        input_tensor = input_tensor.clone().detach().requires_grad_(True)

        output = self.model(input_tensor)
        logits = output["mask"] if isinstance(output, dict) else output

        if target_mask is not None:
            score = (torch.sigmoid(logits) * target_mask).sum()
        else:
            score = torch.sigmoid(logits).sum()

        self.model.zero_grad()
        score.backward(retain_graph=False)

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=(IMAGE_SIZE, IMAGE_SIZE),
            mode="bilinear",
            align_corners=False
        )

        cam = cam.squeeze().detach().cpu().numpy()

        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


def get_gradcam_target_layer(model):
    return model.dec0.conv[3]


def get_prediction(model, image_tensor):
    model.eval()

    with torch.no_grad(), autocast(device_type="cuda", enabled=(device.type == "cuda")):
        output = model(image_tensor)

    logits = output["mask"] if isinstance(output, dict) else output
    pred_mask = (torch.sigmoid(logits).squeeze().cpu().numpy() > BINARY_THRESHOLD).astype(float)

    return pred_mask


print("✓ GradCAM utilities ready")

✓ GradCAM utilities ready


In [24]:
# ============================================================
# Cell 11 — GradCAM comparison for one dataset
# ============================================================
random.seed(200)
DATASET_FOR_GRADCAM = "TGIF"  # options: TGIF, AutoSplice, CASIA v2, IMD2020

dataset = datasets.get(DATASET_FOR_GRADCAM, None)

if dataset is None or len(dataset) == 0:
    print(f"⚠ No samples for {DATASET_FOR_GRADCAM}")
else:
    unet_model, _ = load_model_by_name("baseline")
    novelty_model, _ = load_model_by_name("novelty")

    gradcam_unet = GradCAM(unet_model, get_gradcam_target_layer(unet_model))
    gradcam_novelty = GradCAM(novelty_model, get_gradcam_target_layer(novelty_model))

    n_samples = min(GRADCAM_SAMPLES, len(dataset))
    indices = random.sample(range(len(dataset)), n_samples)

    fig, axes = plt.subplots(n_samples, 6, figsize=(30, 5 * n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = [
        "Input",
        "Ground Truth",
        "Baseline Pred",
        "Baseline GradCAM",
        "Novelty Pred",
        "Novelty GradCAM"
    ]

    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]

        img_tensor = sample["image"].unsqueeze(0).to(device)
        target_mask = sample["mask"].unsqueeze(0).to(device)

        img_display = unnormalize(sample["image"])
        gt = sample["mask"].numpy()[0]

        pred_unet = get_prediction(unet_model, img_tensor)
        pred_novelty = get_prediction(novelty_model, img_tensor)

        cam_unet = gradcam_unet.generate(img_tensor, target_mask=target_mask)
        cam_novelty = gradcam_novelty.generate(img_tensor, target_mask=target_mask)

        axes[row, 0].imshow(img_display)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_unet, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].axis("off")

        axes[row, 3].imshow(img_display)
        axes[row, 3].imshow(cam_unet, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 3].axis("off")

        axes[row, 4].imshow(pred_novelty, cmap="gray", vmin=0, vmax=1)
        axes[row, 4].axis("off")

        axes[row, 5].imshow(img_display)
        axes[row, 5].imshow(cam_novelty, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 5].axis("off")

    safe_name = DATASET_FOR_GRADCAM.lower().replace(" ", "_").replace("-", "_")
    save_path = SAVE_DIR / f"gradcam_comparison_{safe_name}.png"

    fig.suptitle(f"GradCAM Comparison — {DATASET_FOR_GRADCAM}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("✓ Saved:", save_path)

    gradcam_unet.remove_hooks()
    gradcam_novelty.remove_hooks()

    clear_model(unet_model)
    clear_model(novelty_model)

✓ Saved: /home/ubuntu/xai_results/gradcam_comparison_tgif.png


In [22]:
# ============================================================
# Cell 11 — GradCAM comparison for one dataset
# ============================================================

DATASET_FOR_GRADCAM = "AutoSplice"  # options: TGIF, AutoSplice, CASIA v2, IMD2020

dataset = datasets.get(DATASET_FOR_GRADCAM, None)

if dataset is None or len(dataset) == 0:
    print(f"⚠ No samples for {DATASET_FOR_GRADCAM}")
else:
    unet_model, _ = load_model_by_name("baseline")
    novelty_model, _ = load_model_by_name("novelty")

    gradcam_unet = GradCAM(unet_model, get_gradcam_target_layer(unet_model))
    gradcam_novelty = GradCAM(novelty_model, get_gradcam_target_layer(novelty_model))

    n_samples = min(GRADCAM_SAMPLES, len(dataset))
    indices = random.sample(range(len(dataset)), n_samples)

    fig, axes = plt.subplots(n_samples, 6, figsize=(30, 5 * n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = [
        "Input",
        "Ground Truth",
        "Baseline Pred",
        "Baseline GradCAM",
        "Novelty Pred",
        "Novelty GradCAM"
    ]

    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]

        img_tensor = sample["image"].unsqueeze(0).to(device)
        target_mask = sample["mask"].unsqueeze(0).to(device)

        img_display = unnormalize(sample["image"])
        gt = sample["mask"].numpy()[0]

        pred_unet = get_prediction(unet_model, img_tensor)
        pred_novelty = get_prediction(novelty_model, img_tensor)

        cam_unet = gradcam_unet.generate(img_tensor, target_mask=target_mask)
        cam_novelty = gradcam_novelty.generate(img_tensor, target_mask=target_mask)

        axes[row, 0].imshow(img_display)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_unet, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].axis("off")

        axes[row, 3].imshow(img_display)
        axes[row, 3].imshow(cam_unet, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 3].axis("off")

        axes[row, 4].imshow(pred_novelty, cmap="gray", vmin=0, vmax=1)
        axes[row, 4].axis("off")

        axes[row, 5].imshow(img_display)
        axes[row, 5].imshow(cam_novelty, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 5].axis("off")

    safe_name = DATASET_FOR_GRADCAM.lower().replace(" ", "_").replace("-", "_")
    save_path = SAVE_DIR / f"gradcam_comparison_{safe_name}.png"

    fig.suptitle(f"GradCAM Comparison — {DATASET_FOR_GRADCAM}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("✓ Saved:", save_path)

    gradcam_unet.remove_hooks()
    gradcam_novelty.remove_hooks()

    clear_model(unet_model)
    clear_model(novelty_model)

✓ Saved: /home/ubuntu/xai_results/gradcam_comparison_autosplice.png


In [16]:
# ============================================================
# Cell 11 — GradCAM comparison for one dataset
# ============================================================

DATASET_FOR_GRADCAM = "CASIA v2"  # options: TGIF, AutoSplice, CASIA v2, IMD2020

dataset = datasets.get(DATASET_FOR_GRADCAM, None)

if dataset is None or len(dataset) == 0:
    print(f"⚠ No samples for {DATASET_FOR_GRADCAM}")
else:
    unet_model, _ = load_model_by_name("baseline")
    novelty_model, _ = load_model_by_name("novelty")

    gradcam_unet = GradCAM(unet_model, get_gradcam_target_layer(unet_model))
    gradcam_novelty = GradCAM(novelty_model, get_gradcam_target_layer(novelty_model))

    n_samples = min(GRADCAM_SAMPLES, len(dataset))
    indices = random.sample(range(len(dataset)), n_samples)

    fig, axes = plt.subplots(n_samples, 6, figsize=(30, 5 * n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = [
        "Input",
        "Ground Truth",
        "Baseline Pred",
        "Baseline GradCAM",
        "Novelty Pred",
        "Novelty GradCAM"
    ]

    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]

        img_tensor = sample["image"].unsqueeze(0).to(device)
        target_mask = sample["mask"].unsqueeze(0).to(device)

        img_display = unnormalize(sample["image"])
        gt = sample["mask"].numpy()[0]

        pred_unet = get_prediction(unet_model, img_tensor)
        pred_novelty = get_prediction(novelty_model, img_tensor)

        cam_unet = gradcam_unet.generate(img_tensor, target_mask=target_mask)
        cam_novelty = gradcam_novelty.generate(img_tensor, target_mask=target_mask)

        axes[row, 0].imshow(img_display)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_unet, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].axis("off")

        axes[row, 3].imshow(img_display)
        axes[row, 3].imshow(cam_unet, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 3].axis("off")

        axes[row, 4].imshow(pred_novelty, cmap="gray", vmin=0, vmax=1)
        axes[row, 4].axis("off")

        axes[row, 5].imshow(img_display)
        axes[row, 5].imshow(cam_novelty, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 5].axis("off")

    safe_name = DATASET_FOR_GRADCAM.lower().replace(" ", "_").replace("-", "_")
    save_path = SAVE_DIR / f"gradcam_comparison_{safe_name}.png"

    fig.suptitle(f"GradCAM Comparison — {DATASET_FOR_GRADCAM}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("✓ Saved:", save_path)

    gradcam_unet.remove_hooks()
    gradcam_novelty.remove_hooks()

    clear_model(unet_model)
    clear_model(novelty_model)

✓ Saved: /home/ubuntu/xai_results/gradcam_comparison_casia_v2.png


In [19]:
# ============================================================
# Cell 11 — GradCAM comparison for one dataset
# ============================================================

DATASET_FOR_GRADCAM = "IMD2020"  # options: TGIF, AutoSplice, CASIA v2, IMD2020

dataset = datasets.get(DATASET_FOR_GRADCAM, None)

if dataset is None or len(dataset) == 0:
    print(f"⚠ No samples for {DATASET_FOR_GRADCAM}")
else:
    unet_model, _ = load_model_by_name("baseline")
    novelty_model, _ = load_model_by_name("novelty")

    gradcam_unet = GradCAM(unet_model, get_gradcam_target_layer(unet_model))
    gradcam_novelty = GradCAM(novelty_model, get_gradcam_target_layer(novelty_model))

    n_samples = min(GRADCAM_SAMPLES, len(dataset))
    indices = random.sample(range(len(dataset)), n_samples)

    fig, axes = plt.subplots(n_samples, 6, figsize=(30, 5 * n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = [
        "Input",
        "Ground Truth",
        "Baseline Pred",
        "Baseline GradCAM",
        "Novelty Pred",
        "Novelty GradCAM"
    ]

    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]

        img_tensor = sample["image"].unsqueeze(0).to(device)
        target_mask = sample["mask"].unsqueeze(0).to(device)

        img_display = unnormalize(sample["image"])
        gt = sample["mask"].numpy()[0]

        pred_unet = get_prediction(unet_model, img_tensor)
        pred_novelty = get_prediction(novelty_model, img_tensor)

        cam_unet = gradcam_unet.generate(img_tensor, target_mask=target_mask)
        cam_novelty = gradcam_novelty.generate(img_tensor, target_mask=target_mask)

        axes[row, 0].imshow(img_display)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_unet, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].axis("off")

        axes[row, 3].imshow(img_display)
        axes[row, 3].imshow(cam_unet, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 3].axis("off")

        axes[row, 4].imshow(pred_novelty, cmap="gray", vmin=0, vmax=1)
        axes[row, 4].axis("off")

        axes[row, 5].imshow(img_display)
        axes[row, 5].imshow(cam_novelty, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        axes[row, 5].axis("off")

    safe_name = DATASET_FOR_GRADCAM.lower().replace(" ", "_").replace("-", "_")
    save_path = SAVE_DIR / f"gradcam_comparison_{safe_name}.png"

    fig.suptitle(f"GradCAM Comparison — {DATASET_FOR_GRADCAM}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("✓ Saved:", save_path)

    gradcam_unet.remove_hooks()
    gradcam_novelty.remove_hooks()

    clear_model(unet_model)
    clear_model(novelty_model)

✓ Saved: /home/ubuntu/xai_results/gradcam_comparison_imd2020.png


In [25]:
# ============================================================
# Cell 12 — Faithfulness metrics
# ============================================================

def cam_inside_mask_ratio(cam, mask):
    cam = cam.astype(np.float32)
    mask = mask.astype(np.float32)
    return float((cam * mask).sum() / (cam.sum() + 1e-8))


def compute_faithfulness(model, model_key, dataset, dataset_name, n_samples=FAITHFULNESS_SAMPLES):
    if dataset is None or len(dataset) == 0:
        return None

    target_layer = get_gradcam_target_layer(model)
    gradcam = GradCAM(model, target_layer)

    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))

    correlations = []
    inside_ratios = []

    for idx in tqdm(indices, desc=f"Faithfulness {model_key} on {dataset_name}", leave=False):
        sample = dataset[idx]

        img_tensor = sample["image"].unsqueeze(0).to(device)
        target_mask = sample["mask"].unsqueeze(0).to(device)
        mask_gt = sample["mask"].numpy()[0]

        if mask_gt.sum() < 100:
            continue

        cam = gradcam.generate(img_tensor, target_mask=target_mask)

        if cam.shape != mask_gt.shape:
            mask_resized = cv2.resize(
                mask_gt,
                (cam.shape[1], cam.shape[0]),
                interpolation=cv2.INTER_NEAREST
            )
        else:
            mask_resized = mask_gt

        cam_flat = cam.flatten()
        mask_flat = mask_resized.flatten()

        if cam_flat.std() > 0 and mask_flat.std() > 0:
            corr = np.corrcoef(cam_flat, mask_flat)[0, 1]
            if not np.isnan(corr):
                correlations.append(corr)

        inside_ratios.append(cam_inside_mask_ratio(cam, mask_resized))

    gradcam.remove_hooks()

    return {
        "dataset": dataset_name,
        "model": model_key,
        "mean_correlation": float(np.mean(correlations)) if correlations else 0.0,
        "std_correlation": float(np.std(correlations)) if correlations else 0.0,
        "mean_cam_inside_mask": float(np.mean(inside_ratios)) if inside_ratios else 0.0,
        "std_cam_inside_mask": float(np.std(inside_ratios)) if inside_ratios else 0.0,
        "n_evaluated": len(inside_ratios),
    }


all_faithfulness_results = {}

for dataset_name, dataset in datasets.items():
    if dataset is None or len(dataset) == 0:
        continue

    all_faithfulness_results[dataset_name] = {}

    # Baseline
    model, model_name = load_model_by_name("baseline")
    faith_base = compute_faithfulness(model, model_name, dataset, dataset_name)
    all_faithfulness_results[dataset_name][model_name] = faith_base
    clear_model(model)

    # Novelty
    model, model_name = load_model_by_name("novelty")
    faith_novelty = compute_faithfulness(model, model_name, dataset, dataset_name)
    all_faithfulness_results[dataset_name][model_name] = faith_novelty
    clear_model(model)

    print(f"\n{dataset_name}")
    print("Baseline:", faith_base)
    print("Novelty :", faith_novelty)

with open(SAVE_DIR / "faithfulness_results.json", "w") as f:
    json.dump(all_faithfulness_results, f, indent=2, default=str)

print("✓ Saved:", SAVE_DIR / "faithfulness_results.json")


TGIF
Baseline: {'dataset': 'TGIF', 'model': 'ConvNeXt-L U-Net baseline', 'mean_correlation': 0.7760557621625909, 'std_correlation': 0.17679770241139403, 'mean_cam_inside_mask': 0.43430926129221914, 'std_cam_inside_mask': 0.3913056950747556, 'n_evaluated': 10}
Novelty : {'dataset': 'TGIF', 'model': 'Full novelty augmented', 'mean_correlation': 0.677206522811115, 'std_correlation': 0.36012982798465104, 'mean_cam_inside_mask': 0.6019308477640152, 'std_cam_inside_mask': 0.35898621338847053, 'n_evaluated': 10}



AutoSplice
Baseline: {'dataset': 'AutoSplice', 'model': 'ConvNeXt-L U-Net baseline', 'mean_correlation': 0.16940468807215014, 'std_correlation': 0.2904812167621155, 'mean_cam_inside_mask': 0.13707516007125378, 'std_cam_inside_mask': 0.2821868246282749, 'n_evaluated': 10}
Novelty : {'dataset': 'AutoSplice', 'model': 'Full novelty augmented', 'mean_correlation': -0.013893945649278226, 'std_correlation': 0.2086167990272592, 'mean_cam_inside_mask': 0.16766202458820773, 'std_cam_inside_mask': 0.32789944511897057, 'n_evaluated': 10}



CASIA v2
Baseline: {'dataset': 'CASIA v2', 'model': 'ConvNeXt-L U-Net baseline', 'mean_correlation': 0.05908738726925055, 'std_correlation': 0.2522158203976135, 'mean_cam_inside_mask': 0.00787856571841985, 'std_cam_inside_mask': 0.020624923708088598, 'n_evaluated': 10}
Novelty : {'dataset': 'CASIA v2', 'model': 'Full novelty augmented', 'mean_correlation': -0.032395830741253476, 'std_correlation': 0.029431703364622995, 'mean_cam_inside_mask': 0.0, 'std_cam_inside_mask': 0.0, 'n_evaluated': 10}



IMD2020
Baseline: {'dataset': 'IMD2020', 'model': 'ConvNeXt-L U-Net baseline', 'mean_correlation': -0.07675658897426954, 'std_correlation': 0.09536791845692535, 'mean_cam_inside_mask': 0.07286942853695816, 'std_cam_inside_mask': 0.20197082995056415, 'n_evaluated': 9}
Novelty : {'dataset': 'IMD2020', 'model': 'Full novelty augmented', 'mean_correlation': -0.06788616551048463, 'std_correlation': 0.07090325365408223, 'mean_cam_inside_mask': 0.03541957546640333, 'std_cam_inside_mask': 0.10477155799754521, 'n_evaluated': 10}
✓ Saved: /home/ubuntu/xai_results/faithfulness_results.json


In [18]:
# ============================================================
# Cell 13 — Final save + zip
# ============================================================

final_results = {
    "cross_domain": combined_results if "combined_results" in globals() else None,
    "faithfulness": all_faithfulness_results if "all_faithfulness_results" in globals() else None,
    "models": {
        "best_model": {
            "name": "Full novelty augmented",
            "checkpoint": str(novelty_ckpt_path),
            "known_test_iou": 0.8742,
        },
        "baseline": {
            "name": "ConvNeXt-L U-Net baseline",
            "checkpoint": str(baseline_ckpt_path),
        },
    },
    "settings": {
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "binary_threshold": BINARY_THRESHOLD,
        "gradcam_samples": GRADCAM_SAMPLES,
        "faithfulness_samples": FAITHFULNESS_SAMPLES,
    }
}

final_json = SAVE_DIR / "final_xai_cross_domain_results.json"

with open(final_json, "w") as f:
    json.dump(final_results, f, indent=2, default=str)

zip_path = shutil.make_archive(
    base_name=str(HOME / "xai_results_backup"),
    format="zip",
    root_dir=SAVE_DIR
)

print("✓ Final JSON:", final_json)
print("✓ ZIP:", zip_path)

print("\nSaved files:")
for p in sorted(SAVE_DIR.glob("*")):
    print(" ", p.name, f"{p.stat().st_size/1024:.1f} KB")

✓ Final JSON: /home/ubuntu/xai_results/final_xai_cross_domain_results.json
✓ ZIP: /home/ubuntu/xai_results_backup.zip

Saved files:
  baseline_cross_domain_results.json 1.4 KB
  cross_domain_combined_results.json 3.7 KB
  faithfulness_results.json 2.6 KB
  final_xai_cross_domain_results.json 7.3 KB
  gradcam_comparison_autosplice.png 2118.4 KB
  gradcam_comparison_casia_v2.png 3549.9 KB
  gradcam_comparison_tgif.png 3834.6 KB
  novelty_cross_domain_results.json 1.4 KB


In [6]:
# ============================================================
# New Cell A — Download extra AI/inpainting datasets
# 1) Inpainting Localization Evaluation Dataset
# 2) DeFacto Inpainting
# ============================================================

from pathlib import Path
import zipfile, os, json, shutil

HOME = Path.home()
ASSET_ROOT = HOME / "deepfake_assets"
EXTRA_ROOT = ASSET_ROOT / "extra_ai_forgery"

EXTRA_ROOT.mkdir(parents=True, exist_ok=True)

INPAINT_EVAL_ROOT = EXTRA_ROOT / "inpainting_localization_eval"
DEFACTO_ROOT = EXTRA_ROOT / "defacto_inpainting"

# ----------------------------
# Dataset 1: Inpainting Localization Evaluation Dataset
# Kaggle slug: duyminhle/inpainting-localization-eval-dataset
# ----------------------------
if not INPAINT_EVAL_ROOT.exists() or not any(INPAINT_EVAL_ROOT.iterdir()):
    print("Downloading Inpainting Localization Evaluation Dataset...")
    
    !kaggle datasets download -d duyminhle/inpainting-localization-eval-dataset -p "{EXTRA_ROOT}" -q

    zip_path = EXTRA_ROOT / "inpainting-localization-eval-dataset.zip"
    INPAINT_EVAL_ROOT.mkdir(parents=True, exist_ok=True)

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(INPAINT_EVAL_ROOT)
        zip_path.unlink()
        print("✓ Extracted:", INPAINT_EVAL_ROOT)
    else:
        print("⚠ Zip not found. Check Kaggle slug or authentication.")
else:
    print("✓ Already exists:", INPAINT_EVAL_ROOT)


# ----------------------------
# Dataset 2: DeFacto Inpainting
# Kaggle slug: defactodataset/defactoinpainting
# ----------------------------
if not DEFACTO_ROOT.exists() or not any(DEFACTO_ROOT.iterdir()):
    print("\nDownloading DeFacto Inpainting...")
    
    !kaggle datasets download -d defactodataset/defactoinpainting -p "{EXTRA_ROOT}" -q

    zip_path = EXTRA_ROOT / "defactoinpainting.zip"
    DEFACTO_ROOT.mkdir(parents=True, exist_ok=True)

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DEFACTO_ROOT)
        zip_path.unlink()
        print("✓ Extracted:", DEFACTO_ROOT)
    else:
        print("⚠ Zip not found. Check Kaggle slug or authentication.")
else:
    print("✓ Already exists:", DEFACTO_ROOT)


print("\nExtra dataset roots:")
print("INPAINT_EVAL_ROOT:", INPAINT_EVAL_ROOT, INPAINT_EVAL_ROOT.exists())
print("DEFACTO_ROOT     :", DEFACTO_ROOT, DEFACTO_ROOT.exists())

print("\nTop-level Inpainting Eval:")
if INPAINT_EVAL_ROOT.exists():
    for p in list(INPAINT_EVAL_ROOT.iterdir())[:30]:
        print(" ", p)

print("\nTop-level DeFacto:")
if DEFACTO_ROOT.exists():
    for p in list(DEFACTO_ROOT.iterdir())[:30]:
        print(" ", p)

Dataset URL: https://www.kaggle.com/datasets/duyminhle/inpainting-localization-eval-dataset
License(s): CC0-1.0
✓ Extracted: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval

Dataset URL: https://www.kaggle.com/datasets/defactodataset/defactoinpainting
License(s): unknown
✓ Extracted: /home/ubuntu/deepfake_assets/extra_ai_forgery/defacto_inpainting

Extra dataset roots:
INPAINT_EVAL_ROOT: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval True
DEFACTO_ROOT     : /home/ubuntu/deepfake_assets/extra_ai_forgery/defacto_inpainting True

Top-level Inpainting Eval:
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/EC
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/README.md
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/TE
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/NS
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting

In [7]:
# ============================================================
# New Cell B — Safe folder inspection
# ============================================================

def safe_show_dirs(root: Path, max_dirs=80):
    print(f"\nInspecting: {root}")
    count = 0
    
    for p in root.rglob("*"):
        if p.is_dir():
            print(" ", p)
            count += 1
            
            if count >= max_dirs:
                print(f"  ... showing first {max_dirs} dirs only")
                break


def count_images_quick(root: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    n = 0
    
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            n += 1
    
    return n


safe_show_dirs(INPAINT_EVAL_ROOT, max_dirs=80)
safe_show_dirs(DEFACTO_ROOT, max_dirs=80)

print("\nImage counts:")
print("Inpainting Eval:", count_images_quick(INPAINT_EVAL_ROOT))
print("DeFacto        :", count_images_quick(DEFACTO_ROOT))


Inspecting: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/EC
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/TE
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/NS
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/LR
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/LB
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/RN
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/CA
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/SH
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/GC
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/SG
  /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/PM

Inspecting: /home/ubuntu/deepfa

In [8]:
# ============================================================
# New Cell C — Generic indexer for extra AI-forgery datasets
# ============================================================

def build_extra_ai_forgery_index(root: Path, dataset_name: str, max_samples=1000):
    """
    Flexible image-mask matcher for AI/inpainting datasets.
    Designed for datasets where exact folder layout may vary.

    Strategy:
    1. Collect all image-like files.
    2. Treat files/folders containing mask/gt/groundtruth/label as mask candidates.
    3. Match images to masks by normalized filename stem.
    4. If no mask-like paths are found, use binary-mask detection as fallback.
    """
    samples = []

    if not root.exists():
        print(f"⚠ Missing root: {root}")
        return samples

    all_image_files = [
        p for p in root.rglob("*")
        if p.is_file() and is_image_file(p)
    ]

    mask_files = [
        p for p in all_image_files
        if is_mask_like_path(p)
    ]

    # Fallback: binary mask detection, but capped to avoid excessive scanning
    if len(mask_files) == 0:
        print(f"⚠ No obvious mask paths for {dataset_name}. Trying binary-mask fallback...")
        candidate_pool = all_image_files[:5000]
        mask_files = [p for p in candidate_pool if is_probably_binary_mask(p)]

    mask_set = set(mask_files)

    image_files = [
        p for p in all_image_files
        if p not in mask_set
    ]

    print(f"\n{dataset_name}")
    print(f"  total image-like files : {len(all_image_files)}")
    print(f"  mask candidates        : {len(mask_files)}")
    print(f"  image candidates       : {len(image_files)}")

    for img_path in tqdm(image_files, desc=f"Indexing {dataset_name}", leave=False):
        mask_path = find_matching_mask(img_path, mask_files)

        if mask_path is not None:
            samples.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path),
                "is_fake": True,
                "dataset": dataset_name
            })

            if max_samples is not None and len(samples) >= max_samples:
                break

    print(f"  paired samples         : {len(samples)}")

    return samples


# Start with 1000 samples for stability.
# Later you can set max_samples=None if the structure is clean and time permits.
inpaint_eval_samples = build_extra_ai_forgery_index(
    INPAINT_EVAL_ROOT,
    dataset_name="Inpainting Eval",
    max_samples=1000
)

defacto_samples = build_extra_ai_forgery_index(
    DEFACTO_ROOT,
    dataset_name="DeFacto Inpainting",
    max_samples=1000
)

datasets["Inpainting Eval"] = GenericDataset(inpaint_eval_samples, forged_only=True) if inpaint_eval_samples else None
datasets["DeFacto Inpainting"] = GenericDataset(defacto_samples, forged_only=True) if defacto_samples else None

print("\nUpdated dataset registry:")
for name, ds in datasets.items():
    print(f"{name:<24}: {len(ds) if ds is not None else 0}")


Inpainting Eval
  total image-like files : 22000
  mask candidates        : 11000
  image candidates       : 11000


  paired samples         : 1000

DeFacto Inpainting
  total image-like files : 75003
  mask candidates        : 50002
  image candidates       : 25001


  paired samples         : 1000

Updated dataset registry:
TGIF                    : 686
AutoSplice              : 3621
CASIA v2                : 5123
IMD2020                 : 300
Inpainting Eval         : 1000
DeFacto Inpainting      : 1000


In [11]:
# ============================================================
# Inspect sample filenames for extra datasets
# ============================================================

from pathlib import Path

def show_sample_files(root: Path, max_files=20):
    print(f"\nInspecting files under: {root}")
    files = [p for p in root.rglob("*") if p.is_file() and is_image_file(p)]
    print("Total image-like files:", len(files))
    for p in files[:max_files]:
        print(" ", p.relative_to(root))

print("Inpainting Eval sample files:")
for method_dir in sorted([p for p in INPAINT_EVAL_ROOT.iterdir() if p.is_dir()])[:3]:
    show_sample_files(method_dir, max_files=20)

print("\nDeFacto image samples:")
show_sample_files(DEFACTO_ROOT / "inpainting_img" / "img", max_files=10)

print("\nDeFacto inpaint mask samples:")
show_sample_files(DEFACTO_ROOT / "inpainting_annotations" / "inpaint_mask", max_files=10)

print("\nDeFacto probe mask samples:")
show_sample_files(DEFACTO_ROOT / "inpainting_annotations" / "probe_mask", max_files=10)

Inpainting Eval sample files:

Inspecting files under: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/CA
Total image-like files: 2000
  000109_mask.png
  000195_mask.png
  000324.png
  000225_mask.png
  000601.png
  000960_mask.png
  000548.png
  000177.png
  000378.png
  000764_mask.png
  000933_mask.png
  000719.png
  000332.png
  000662.png
  000729_mask.png
  000658_mask.png
  000620_mask.png
  000788.png
  000135_mask.png
  000375.png

Inspecting files under: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localization_eval/EC
Total image-like files: 2000
  002829.png
  002680_mask.png
  002492_mask.png
  002218.png
  002613.png
  002658_mask.png
  002225.png
  002668.png
  002456_mask.png
  002301.png
  002493_mask.png
  002190.png
  002193_mask.png
  002015_mask.png
  002408.png
  002034_mask.png
  002826.png
  002641.png
  002267_mask.png
  002593.png

Inspecting files under: /home/ubuntu/deepfake_assets/extra_ai_forgery/inpainting_localizat

In [12]:
# ============================================================
# New Cell C — Clean indexers for Inpainting Eval + DeFacto
# ============================================================

# ------------------------------------------------------------
# Inpainting Localization Evaluation Dataset
# ------------------------------------------------------------
def build_inpainting_eval_index_clean(root: Path, max_per_method=200):
    """
    Structure:
      root/
        CA/
          000324.png
          000324_mask.png
        EC/
          002829.png
          002829_mask.png
        ...

    Pair rule:
      <id>.png -> <id>_mask.png
    """
    samples = []

    method_dirs = sorted([p for p in root.iterdir() if p.is_dir()])

    print("\nInpainting Localization Eval")
    print("  method folders:", [p.name for p in method_dirs])

    for method_dir in method_dirs:
        image_files = sorted([
            p for p in method_dir.glob("*.png")
            if not p.name.lower().endswith("_mask.png")
        ])

        method_count = 0

        for img_path in image_files:
            mask_path = img_path.with_name(f"{img_path.stem}_mask.png")

            if not mask_path.exists():
                continue

            samples.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path),
                "is_fake": True,
                "dataset": f"Inpainting Eval_{method_dir.name}",
                "method": method_dir.name
            })

            method_count += 1

            if max_per_method is not None and method_count >= max_per_method:
                break

        print(f"  {method_dir.name:<4}: paired {method_count}")

    print("  total paired samples:", len(samples))
    return samples


# ------------------------------------------------------------
# DeFacto Inpainting
# ------------------------------------------------------------
def build_defacto_inpainting_index_clean(root: Path, max_samples=1000):
    """
    Structure:
      root/
        inpainting_img/img/<name>.tif
        inpainting_annotations/inpaint_mask/<name>.tif
        inpainting_annotations/probe_mask/<name>.tif

    Use inpaint_mask for localization.
    """
    samples = []

    img_dir = root / "inpainting_img" / "img"
    mask_dir = root / "inpainting_annotations" / "inpaint_mask"

    if not img_dir.exists():
        print("⚠ Missing DeFacto image dir:", img_dir)
        return samples

    if not mask_dir.exists():
        print("⚠ Missing DeFacto mask dir:", mask_dir)
        return samples

    image_files = sorted([
        p for p in img_dir.rglob("*")
        if p.is_file() and is_image_file(p)
    ])

    count = 0

    for img_path in tqdm(image_files, desc="Indexing DeFacto", leave=False):
        rel = img_path.relative_to(img_dir)
        mask_path = mask_dir / rel

        if not mask_path.exists():
            continue

        samples.append({
            "image_path": str(img_path),
            "mask_path": str(mask_path),
            "is_fake": True,
            "dataset": "DeFacto Inpainting"
        })

        count += 1

        if max_samples is not None and count >= max_samples:
            break

    print("\nDeFacto Inpainting")
    print("  paired samples:", len(samples))

    return samples


# Build datasets
inpaint_eval_samples = build_inpainting_eval_index_clean(
    INPAINT_EVAL_ROOT,
    max_per_method=200   # 11 folders × 200 = up to 2200 samples
)

defacto_samples = build_defacto_inpainting_index_clean(
    DEFACTO_ROOT,
    max_samples=1000
)

datasets["Inpainting Eval"] = GenericDataset(inpaint_eval_samples, forged_only=True) if inpaint_eval_samples else None
datasets["DeFacto Inpainting"] = GenericDataset(defacto_samples, forged_only=True) if defacto_samples else None

print("\nUpdated dataset registry:")
for name, ds in datasets.items():
    print(f"{name:<24}: {len(ds) if ds is not None else 0}")


Inpainting Localization Eval
  method folders: ['CA', 'EC', 'GC', 'LB', 'LR', 'NS', 'PM', 'RN', 'SG', 'SH', 'TE']
  CA  : paired 200
  EC  : paired 200
  GC  : paired 200
  LB  : paired 200
  LR  : paired 200
  NS  : paired 200
  PM  : paired 200
  RN  : paired 200
  SG  : paired 200
  SH  : paired 200
  TE  : paired 200
  total paired samples: 2200



DeFacto Inpainting
  paired samples: 1000

Updated dataset registry:
TGIF                    : 686
AutoSplice              : 3621
CASIA v2                : 5123
IMD2020                 : 300
Inpainting Eval         : 2200
DeFacto Inpainting      : 1000


In [14]:
show_dataset_pairs("Inpainting Eval", n=4)
show_dataset_pairs("DeFacto Inpainting", n=4)

In [15]:
EXTRA_DATASET_NAMES = ["Inpainting Eval", "DeFacto Inpainting"]

model, model_name = load_model_by_name("novelty")

extra_novelty_results = {}

for dataset_name in EXTRA_DATASET_NAMES:
    dataset = datasets.get(dataset_name, None)

    if dataset is None or len(dataset) == 0:
        print(f"⚠ Skipping {dataset_name}: no samples")
        continue

    metrics = evaluate_model_streaming(
        model=model,
        dataset=dataset,
        dataset_name=dataset_name,
        model_name=model_name,
        max_samples=1000
    )

    extra_novelty_results[dataset_name] = metrics

    print(
        f"{dataset_name:<24} | {model_name:<28} | "
        f"IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | "
        f"Prec: {metrics['precision']:.4f} | Rec: {metrics['recall']:.4f} | "
        f"N: {metrics['n_samples']}"
    )

with open(SAVE_DIR / "extra_ai_forgery_novelty_results.json", "w") as f:
    json.dump(extra_novelty_results, f, indent=2, default=str)

print("✓ Saved:", SAVE_DIR / "extra_ai_forgery_novelty_results.json")

clear_model(model)

Inpainting Eval          | Full novelty augmented       | IoU: 0.1314 | F1: 0.2323 | Prec: 0.9286 | Rec: 0.1328 | N: 1000


DeFacto Inpainting       | Full novelty augmented       | IoU: 0.0065 | F1: 0.0128 | Prec: 0.2711 | Rec: 0.0066 | N: 1000
✓ Saved: /home/ubuntu/xai_results/extra_ai_forgery_novelty_results.json


In [16]:
EXTRA_DATASET_NAMES = ["Inpainting Eval", "DeFacto Inpainting"]

model, model_name = load_model_by_name("baseline")

extra_baseline_results = {}

for dataset_name in EXTRA_DATASET_NAMES:
    dataset = datasets.get(dataset_name, None)

    if dataset is None or len(dataset) == 0:
        print(f"⚠ Skipping {dataset_name}: no samples")
        continue

    metrics = evaluate_model_streaming(
        model=model,
        dataset=dataset,
        dataset_name=dataset_name,
        model_name=model_name,
        max_samples=1000
    )

    extra_baseline_results[dataset_name] = metrics

    print(
        f"{dataset_name:<24} | {model_name:<28} | "
        f"IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | "
        f"Prec: {metrics['precision']:.4f} | Rec: {metrics['recall']:.4f} | "
        f"N: {metrics['n_samples']}"
    )

with open(SAVE_DIR / "extra_ai_forgery_baseline_results.json", "w") as f:
    json.dump(extra_baseline_results, f, indent=2, default=str)

print("✓ Saved:", SAVE_DIR / "extra_ai_forgery_baseline_results.json")

clear_model(model)

Inpainting Eval          | ConvNeXt-L U-Net baseline    | IoU: 0.0800 | F1: 0.1481 | Prec: 0.9220 | Rec: 0.0805 | N: 1000


DeFacto Inpainting       | ConvNeXt-L U-Net baseline    | IoU: 0.0115 | F1: 0.0228 | Prec: 0.2189 | Rec: 0.0120 | N: 1000
✓ Saved: /home/ubuntu/xai_results/extra_ai_forgery_baseline_results.json
